# vgra-yolo background training (Save Version)

One notebook per Kaggle account. It trains up to two runs at once, one per GPU.

- **Version 1:** set `REF` and `RUNS`, then Save Version (Save & Run All) with GPU T4 x2 and Internet on.
- **Version 2, 3, ...:** attach the output of the previous version as input (Add Input), change nothing else, Save Version again.
  Each run continues from where it stopped. A finished run is not trained again.

Each version stops cleanly before the 12-hour limit (time budget `train.max_hours` in `configs/base.yaml`).
Inputs needed: the image dataset and `vgra-index`.

In [ ]:
# Settings: keep these the same in every version of this notebook
REPO = "https://github.com/nithoukhan1/vgra-yolo.git"
REF = "detectors-v1"  # fixed tag, so every version runs exactly the same code
RUNS = [  # (config, seed, GPU)
    ("configs/B0.yaml", 0, 0),
    ("configs/B1.yaml", 0, 1),
]

In [ ]:
# Clone the fixed code, install, check
%cd /kaggle/working
!rm -rf vgra-yolo
!git clone --quiet {REPO} vgra-yolo
%cd /kaggle/working/vgra-yolo
!git checkout --quiet {REF}
!git log -1 --oneline
!pip install -q -r requirements.txt
!pip install -q -e .
!python scripts/00_check_setup.py

In [ ]:
# Train: one process per GPU, each with its own log; wait for both
import os, subprocess, sys, time
from datetime import datetime

os.makedirs("/kaggle/working/logs", exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M")
procs = []
for cfg, seed, gpu in RUNS:
    name = f"{os.path.splitext(os.path.basename(cfg))[0]}_s{seed}"
    log_path = f"/kaggle/working/logs/{name}_{stamp}.log"
    cmd = [sys.executable, "scripts/03_train_detector.py", "--config", cfg, "--seed", str(seed),
           "--device", str(gpu), "--auto-resume"]
    procs.append((name, log_path, subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)))
    print(f"started {name} on GPU {gpu} | log {log_path}", flush=True)
    time.sleep(60)  # the runs share one data file; do not write it at the same moment
for name, log_path, p in procs:
    print(f"{name} ended with exit code {p.wait()}", flush=True)

In [ ]:
# Summary of this version
import glob, json

for cfg, seed, gpu in RUNS:
    name = f"{os.path.splitext(os.path.basename(cfg))[0]}_s{seed}"
    m = f"/kaggle/working/runs/{name}/metrics.json"
    if os.path.exists(m):
        d = json.load(open(m))
        status = "FINISHED" if d.get("finished") else "STOPPED FOR TIME (attach this output, run the next version)"
        print(f"{name}: {status} | epochs {d.get('epochs_total')} | "
              f"valid mAP50 {d['valid']['map50']} | mAP50-95 {d['valid']['map50_95']}")
    else:
        logs = sorted(glob.glob(f"/kaggle/working/logs/{name}_*.log"))
        print(f"{name}: no metrics.json. Last lines of {logs[-1] if logs else 'the log'}:")
        if logs:
            print("".join(open(logs[-1]).readlines()[-25:]))